# Memory Store

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn how to use `InMemoryStore` for cross-thread memory, organize data with namespaces, use `store.put` and `store.search`, enable semantic search with `IndexConfig`, and access the store from nodes.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic InMemoryStore Usage

Store and retrieve data using namespaces and keys.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(("users", "alice", "preferences"), "theme", {"value": "dark"})
store.put(("users", "alice", "preferences"), "language", {"value": "en"})
store.put(("users", "bob", "preferences"), "theme", {"value": "light"})

results = store.search(("users", "alice", "preferences"))
for item in results:
    print(f"{item.key}: {item.value}")

## 4. Storing and Searching Memories

Use the store as a knowledge base for user-specific facts.

In [ ]:
store.put(("memories", "user-42"), "fact-1", {"text": "User prefers Python"})
store.put(("memories", "user-42"), "fact-2", {"text": "User is building a chatbot"})
store.put(("memories", "user-42"), "fact-3", {"text": "User works at a startup"})

results = store.search(("memories", "user-42"))
for item in results:
    print(f"{item.key}: {item.value['text']}")

## 5. Semantic Search with IndexConfig

Enable embedding-based retrieval by configuring the store with an embedding model.

In [ ]:
semantic_store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
        "fields": ["text"],
    }
)

semantic_store.put(("knowledge",), "k1", {"text": "LangGraph uses nodes and edges to build agent workflows"})
semantic_store.put(("knowledge",), "k2", {"text": "Python is a popular programming language"})
semantic_store.put(("knowledge",), "k3", {"text": "StateGraph compiles into a runnable application"})
semantic_store.put(("knowledge",), "k4", {"text": "Checkpointers save graph state for persistence"})

results = semantic_store.search(("knowledge",), query="How does LangGraph work?", limit=2)
for item in results:
    print(f"{item.key}: {item.value['text']} (score: {item.score:.3f})")

## 6. Accessing Store from Graph Nodes

Nodes receive the store automatically when compiled with `store=`. Declare a `store` parameter in the node function.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.base import BaseStore
from langchain_openai import ChatOpenAI
from uuid import uuid4

class State(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str

llm = ChatOpenAI(model="gpt-4o-mini")

def memory_chatbot(state: State, *, store: BaseStore) -> dict:
    user_id = state.get("user_id", "default")
    namespace = ("memories", user_id)

    memories = store.search(namespace)
    memory_text = "\n".join(m.value["text"] for m in memories) if memories else "No memories yet."

    response = llm.invoke([
        ("system", f"You are a helpful assistant. Known facts about the user:\n{memory_text}\n\nIf the user shares a new fact about themselves, note it."),
        *state["messages"],
    ])

    last_human = state["messages"][-1].content if state["messages"] else ""
    if any(keyword in last_human.lower() for keyword in ["my name", "i like", "i am", "i work", "i prefer"]):
        store.put(namespace, str(uuid4()), {"text": last_human})

    return {"messages": [response]}

graph = StateGraph(State)
graph.add_node("chatbot", memory_chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

memory_store = InMemoryStore()
checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer, store=memory_store)

In [ ]:
config1 = {"configurable": {"thread_id": "thread-1"}}

result = app.invoke(
    {"messages": [("human", "My name is Alice and I prefer Python")], "user_id": "alice"},
    config=config1,
)
print(f"Thread 1: {result['messages'][-1].content}")

In [ ]:
config2 = {"configurable": {"thread_id": "thread-2"}}

result = app.invoke(
    {"messages": [("human", "What do you know about me?")], "user_id": "alice"},
    config=config2,
)
print(f"Thread 2 (different thread, same user): {result['messages'][-1].content}")

In [ ]:
stored_memories = memory_store.search(("memories", "alice"))
print("Stored memories for Alice:")
for m in stored_memories:
    print(f"  {m.key}: {m.value['text']}")

## What You Learned

1. **`InMemoryStore`** provides cross-thread memory beyond individual conversations
2. **Namespaces** are tuples of strings for hierarchical data organization
3. **`store.put(namespace, key, value)`** writes; **`store.search(namespace)`** reads
4. **Semantic search** with `IndexConfig` finds memories by meaning using embeddings
5. Nodes access the store via a **`store` parameter** — LangGraph injects it automatically